<a href="https://colab.research.google.com/github/Sarthak-Gawri/Basic-Rag-Pipeline-DOcument-reader-/blob/main/MultidocSearcherRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain-community langchain-classic langchain-text-splitters chromadb sentence-transformers pypdf langchain-groq -q

In [2]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY').strip()

In [3]:
from google.colab import files
uploaded = files.upload()
pdf_paths = list(uploaded.keys())
print(f"Uploaded files: {pdf_paths}")

Saving TW ppt final.pdf to TW ppt final (5).pdf
Saving File handling ppt_20260504_190757_0000.pdf to File handling ppt_20260504_190757_0000 (5).pdf
Uploaded files: ['TW ppt final (5).pdf', 'File handling ppt_20260504_190757_0000 (5).pdf']


In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(chunk_size=800,chunk_overlap=100)

all_chunks=[]
for path in pdf_paths:
  loader=PyPDFLoader(path)
  pages=loader.load()
  chunks=splitter.split_documents(pages)
  for chunk in chunks:
    chunk.metadata["source"]=path
  all_chunks.extend(chunks)
  print(f"{path}: {len(chunks)} chunks created")


print(f"\nTotal chunks across all documents: {len(all_chunks)}")

/tmp/ipykernel_24758/1093733699.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


TW ppt final (5).pdf: 29 chunks created
File handling ppt_20260504_190757_0000 (5).pdf: 15 chunks created

Total chunks across all documents: 44


In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(all_chunks, embedding_model)
print("Vector DB created with chunks from both documents!")

/tmp/ipykernel_24758/2195595238.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector DB created with chunks from both documents!


In [6]:
from langchain_groq import ChatGroq

llm=ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

print("Groq LLM Ready")

Groq LLM Ready


In [13]:
from langchain_classic.chains import RetrievalQA

# qa_chain = RetrievalQA.from_chain_type(
#     llm=llm,
#     retriever=vectorstore.as_retriever(search_kwargs={"k": 6}),
#     return_source_documents=True
# )

# question = "What are the major topics covered in these documents?"
# result = qa_chain.invoke({"query": question})

# print("ANSWER:", result["result"])
# print("\n--- Source chunks used ---")
# for doc in result["source_documents"]:
#     print(f"[Source: {doc.metadata['source']}]")
#     print(doc.page_content[:200], "\n---")

In [15]:
# Retrieve top 3 chunks from EACH source separately, then combine

file_handling_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3, "filter": {"source": "File handling ppt_20260504_190757_0000 (5).pdf"}}
)

self_esteem_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3, "filter": {"source": "TW ppt final (5).pdf"}}
)

question = "What are the major topics covered in these documents?"

file_handling_chunks = file_handling_retriever.invoke(question)
self_esteem_chunks = self_esteem_retriever.invoke(question)

combined_chunks = file_handling_chunks + self_esteem_chunks

context_text = "\n\n".join([doc.page_content for doc in combined_chunks])

prompt = f"""Based on the following context from multiple documents, answer the question.

Context:
{context_text}

Question: {question}

Answer:"""

response = llm.invoke(prompt)
print(response.content)

The major topics covered in these documents are:

1. **File Handling in Java**: This topic is covered in the first document, which explains the importance of file handling in Java, different file classes, methods for creating, writing, reading, and deleting files, and the use of streams for efficient data management.

2. **Self-Esteem and Psychological Theories**: This topic is covered in the second document, which discusses various psychological theories of self-esteem, including Sociometer Theory, Terror Management Theory, Self-Determination Theory, and Symbolic Interaction Theory. It also highlights the importance of self-compassion in protecting self-esteem.

These two topics are unrelated to each other, and the documents appear to be from different sources and authors.
